# 02. 부도 예측 모델 - 피처 엔지니어링

**목적**: EDA 결과를 바탕으로 모델 학습에 사용할 피처 준비

**주요 작업**:
1. 결측치 처리 전략
2. 이상치 처리
3. 파생 피처 생성
4. 범주형 피처 인코딩
5. 피처 선택 (상관관계, 중요도 기반)
6. 피처 스케일링
7. 최종 데이터셋 저장

**입력**: `ml/data/raw_data_20210801.parquet`  
**출력**: `ml/data/processed_data_20210801.parquet`

## 1. 환경 설정

In [ ]:
# 기본 라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif

# 시각화 설정
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# pandas 출력 옵션
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✓ 환경 설정 완료")

## 2. 데이터 로드

In [ ]:
# Parquet 파일에서 데이터 로드
data_path = '../data/raw_data_20210801.parquet'

print("데이터 로딩 중...")
df = pd.read_parquet(data_path)
print(f"✓ 데이터 로드 완료: {len(df):,}행 × {len(df.columns)}열")

# 원본 데이터 복사 (비교용)
df_original = df.copy()

# 타겟 변수 확인
print(f"\n부도율: {df['default_yn'].mean() * 100:.2f}%")
print(f"정상: {(df['default_yn'] == 0).sum():,}개")
print(f"부도: {(df['default_yn'] == 1).sum():,}개")

## 3. 결측치 처리

**전략**:
- 결측률 70% 이상: 삭제
- 결측률 10-70%: 중위값 또는 -999 (비율의 경우)
- 결측률 10% 미만: 중위값

In [ ]:
# 결측치 분석
missing_stats = pd.DataFrame({
    '결측수': df.isnull().sum(),
    '결측률(%)': df.isnull().sum() / len(df) * 100
}).sort_values('결측률(%)', ascending=False)

print("="*60)
print(" 결측치 통계")
print("="*60)
print(missing_stats[missing_stats['결측률(%)'] > 0])
print("="*60)

# 결측률 70% 이상 컬럼 삭제
high_missing_cols = missing_stats[missing_stats['결측률(%)'] >= 70].index.tolist()
if len(high_missing_cols) > 0:
    print(f"\n결측률 70% 이상 컬럼 삭제: {high_missing_cols}")
    df = df.drop(columns=high_missing_cols)

# 나머지 결측치: 중위값으로 대체
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        
print(f"\n✓ 결측치 처리 완료")
print(f"전체 결측치: {df.isnull().sum().sum()}개")

## 4. 이상치 처리

**전략**: IQR 기반 이상치를 중위값으로 대체 (삭제하지 않음 - 데이터 손실 방지)

In [ ]:
# 재무비율 컬럼 선택 (이상치가 많이 발생하는 컬럼)
ratio_cols = [col for col in df.columns if any(x in col for x in ['ratio', 'margin', 'roe', 'roa', 'turnover', 'growth'])]

outlier_counts = {}

for col in ratio_cols:
    if col in df.columns and df[col].dtype in [np.float64, np.int64]:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_count = outliers.sum()
        
        if outlier_count > 0:
            outlier_counts[col] = outlier_count
            # 이상치를 중위값으로 대체
            median_val = df[col].median()
            df.loc[outliers, col] = median_val

print("="*60)
print(" 이상치 처리 결과")
print("="*60)
for col, count in sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{col}: {count:,}개 이상치 처리")
print("="*60)
print(f"\n✓ 총 {sum(outlier_counts.values()):,}개 이상치 처리 완료")

## 5. 파생 피처 생성

부도 예측에 유용한 파생 피처 생성:
- 유동성 지표 (현금/부채 비율)
- 수익성 지표 (EBITDA, 이자보상배율)
- 안정성 지표 (순운전자본)
- 성장성 지표 (복합 성장률)

In [ ]:
# 1. 현금/부채 비율
df['cash_to_debt'] = df['operating_cashflow'] / (df['total_debt'] + 1e-6)
df['cash_to_debt'] = df['cash_to_debt'].replace([np.inf, -np.inf], 0)

# 2. EBITDA (영업이익 + 감가상각비, 간소화)
df['ebitda'] = df['operating_income']

# 3. 이자보상배율 (영업이익 / 이자비용)
df['interest_coverage'] = df['operating_income'] / (df['interest_expense'] + 1e-6)
df['interest_coverage'] = df['interest_coverage'].replace([np.inf, -np.inf], 0)

# 4. 순운전자본 (유동자산 - 유동부채)
df['net_working_capital'] = df['current_asset'] - df['current_liability']

# 5. 순운전자본/총자산 비율
df['nwc_to_total_asset'] = df['net_working_capital'] / (df['total_asset'] + 1e-6)
df['nwc_to_total_asset'] = df['nwc_to_total_asset'].replace([np.inf, -np.inf], 0)

# 6. 복합 성장률 (자산 + 매출 성장률 평균)
df['composite_growth'] = (df['total_asset_growth'] + df['revenue_growth']) / 2

# 7. 차입금 의존도 (차입금/총자산)
df['borrowing_ratio'] = df['borrowings'] / (df['total_asset'] + 1e-6)
df['borrowing_ratio'] = df['borrowing_ratio'].replace([np.inf, -np.inf], 0)

# 8. 공공신용정보 이벤트 여부 (999999999 = 이벤트 없음 → 0, 나머지 → 카테고리)
df['has_credit_event_1'] = (df['public_credit_event_1'] != 999999999).astype(int)
df['has_credit_event_2'] = (df['public_credit_event_2'] != 999999999).astype(int)

print("="*60)
print(" 생성된 파생 피처")
print("="*60)
print("1. cash_to_debt - 현금/부채 비율")
print("2. ebitda - 영업이익 (간소화)")
print("3. interest_coverage - 이자보상배율")
print("4. net_working_capital - 순운전자본")
print("5. nwc_to_total_asset - 순운전자본/총자산")
print("6. composite_growth - 복합 성장률")
print("7. borrowing_ratio - 차입금 의존도")
print("8. has_credit_event_1/2 - 공공신용이벤트 여부")
print("="*60)
print(f"\n✓ 총 {len(df.columns)}개 컬럼 (원본 대비 +8개)")

## 6. 범주형 피처 인코딩

In [ ]:
# 범주형 컬럼 인코딩
categorical_cols = ['industry_code', 'audit_status', 'region']

# LabelEncoder 사용
label_encoders = {}
for col in categorical_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f"✓ {col} 인코딩 완료: {len(le.classes_)}개 클래스")

print(f"\n✓ 범주형 피처 인코딩 완료")

## 7. 피처 선택

**기준**:
1. 타겟과 상관관계가 낮은 피처 제거 (|corr| < 0.01)
2. 분산이 거의 없는 피처 제거 (std < 0.01)
3. Mutual Information 기반 중요도 확인

In [ ]:
# 피처와 타겟 분리
X = df.drop(columns=['default_yn', 'base_date', 'public_credit_event_1', 'public_credit_event_2'])
y = df['default_yn']

print(f"초기 피처 개수: {X.shape[1]}개")

# 1. 분산이 거의 없는 피처 제거
low_var_cols = []
for col in X.columns:
    if X[col].std() < 0.01:
        low_var_cols.append(col)

if len(low_var_cols) > 0:
    print(f"\n분산 낮은 피처 제거: {low_var_cols}")
    X = X.drop(columns=low_var_cols)

# 2. 타겟과 상관관계 분석
correlations = X.corrwith(y).abs().sort_values(ascending=False)

print("\n="*60)
print(" 타겟과 상관관계 Top 20")
print("="*60)
print(correlations.head(20))
print("="*60)

# 상관관계 낮은 피처 제거 (|corr| < 0.01)
low_corr_cols = correlations[correlations < 0.01].index.tolist()
if len(low_corr_cols) > 0:
    print(f"\n상관관계 낮은 피처 제거 ({len(low_corr_cols)}개)")
    X = X.drop(columns=low_corr_cols)

print(f"\n✓ 최종 피처 개수: {X.shape[1]}개")

## 8. 피처 스케일링

**전략**: StandardScaler (평균 0, 분산 1)

In [ ]:
# StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("✓ 피처 스케일링 완료")
print(f"평균: {X_scaled.mean().mean():.4f}")
print(f"표준편차: {X_scaled.std().mean():.4f}")

## 9. 최종 데이터셋 저장

In [ ]:
# 최종 데이터셋 생성 (피처 + 타겟)
final_df = X_scaled.copy()
final_df['default_yn'] = y

# Parquet 저장
output_path = '../data/processed_data_20210801.parquet'
final_df.to_parquet(output_path, index=False, compression='snappy')

print("="*60)
print(" 최종 데이터셋 저장 완료")
print("="*60)
print(f"출력 파일: {output_path}")
print(f"행 수: {len(final_df):,}")
print(f"피처 개수: {len(final_df.columns) - 1}")
print(f"타겟 변수: default_yn")
print(f"부도율: {final_df['default_yn'].mean() * 100:.2f}%")
print("="*60)

# 피처 목록 저장
feature_list = X_scaled.columns.tolist()
print("\n최종 피처 목록:")
for i, feature in enumerate(feature_list, 1):
    print(f"{i:2d}. {feature}")

## 10. 요약

**피처 엔지니어링 완료**

1. ✓ 결측치 처리 (70% 이상 삭제, 나머지 중위값 대체)
2. ✓ 이상치 처리 (IQR 기반 중위값 대체)
3. ✓ 파생 피처 생성 (8개)
4. ✓ 범주형 인코딩 (LabelEncoder)
5. ✓ 피처 선택 (상관관계 기반)
6. ✓ 피처 스케일링 (StandardScaler)

**다음 단계**:
- 모델 학습: `ml/scripts/train_default_model.py`
- 모델 평가 및 SHAP 분석